# NHCS Scenario 4 — Who dies in hospital? Inpatient mortality 2020 to 2022

_A mortality model on half a million discharges. Inpatient dataset only. Level: second analysis, one model. About 90 minutes._

**Before you start.** The NHCS pathology holds two datasets from about 200 US hospitals, 2020 to
2022: 1,368,648 emergency department visit records and 488,636 inpatient discharge records, each a
5% sample of all encounters of the participating hospitals. **A row is an encounter record, not a
patient**, and the two datasets cannot be linked: each is an independent sample, the identifiers
were renumbered, and NCHS merged the ED outcome "admitted" into "other". The MIP does not use the
survey weights: every figure describes *the sampled encounters*, not US hospital care. Codes are
NCHS codes: Sex 1 = Male, 2 = Female; NCHS flags 1 = Yes, 2 = No; curated flags 0 = No, 1 = Yes;
age is top-coded at 86 and the length of stay at 15 days.

**How this notebook works.** Every analysis is a *federated* call: the code selects a dataset and
variables, defines a cohort with filters and asks the MIP for aggregates (counts, means, a test, a
model). No record ever reaches this notebook. Run the cells in order; each step says what to look
for and what result to expect. These are the largest datasets of this MIP: the notebook prints how
long the regressions take.

**Learning goals.** Define a cohort with a filter (no newborns); describe in-hospital mortality by
year, age, sex, comorbidity and diagnosis; read a dramatic change in the case fatality of COVID-19
across three years; compare ages with a t-test; and fit a logistic regression whose odds ratios a
clinician recognises.

**Clinical background.** About one hospital stay in forty ends in death. The risk is concentrated:
it rises with age and with the number and severity of chronic conditions, which the Charlson
comorbidity index summarises in one number, and it depends on why the patient was admitted: sepsis
and respiratory failure are the deadliest reasons. In 2020 COVID-19 arrived as a new cause of
admission with a high case fatality, which fell over the following two years with vaccination,
treatment and the Omicron variant. Newborn stays (10% of discharges) are a population of their own
and are excluded. "Died" is death during the stay; what happened after discharge is unknown.

## 1. Connect to the MIP and open the NHCS pathology

In [ ]:
import mip
from mip.filters import F

client = mip.Client.from_env()          # connection configured by the MIP portal
catalog = client.catalog()

try:
    dm = catalog.data_model("NHCS")        # add version="ED_Inpatient_v1" if the catalogue holds several versions
except LookupError:
    display(catalog.summaries())           # the pathology label differs: pick it from this list
    raise
def dataset(code, label):
    # by the code, which is how the platform labels its datasets (seen 2026-10-12), else by the long label
    try:
        return dm.datasets[code]
    except (KeyError, LookupError):
        return dm.datasets[label]

ed = dataset("nhcs_ed", "NHCS emergency department visits, USA, 2020-2022")
ip = dataset("nhcs_ip", "NHCS inpatient discharges, USA, 2020-2022")
display(dm)
display(dm.datasets.to_frame())

In [ ]:
# The variable tree is the anatomy of one encounter record: Encounter, Patient, Discharge,
# Diagnoses (with the curated condition flags and the NCHS CCSR flags), and the branch
# Inpatient stay (empty for ED visits): length of stay and procedures.
dm.tree()

## 2. Toolbox

In [ ]:
# ---- Toolbox: small helpers that turn MIP results into readable tables and charts. ----
# Run this cell once. You do not need to read it to follow the scenario; come back to it
# when you want to see how a result payload is unpacked.
import math
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from mip.preprocessing import MissingValuesHandler

POOLED = "all datasets"                 # the row describe() adds across the selected datasets
PCT = "{:.1f}"                          # number format of the percentage columns


def analysis(variables, filters=None, drop_missing=False, datasets=None):
    """A Pipeline on the datasets of this notebook (DATASETS, or the ones given) with the given
    variables and cohort filter. Tests and models need drop_missing=True: the platform requires an
    explicit missing-values step and analyses complete cases only. describe() keeps it off so it
    can count missing values."""
    handler = MissingValuesHandler(strategies={v: "drop" for v in variables}) if drop_missing else None
    return mip.Pipeline(analysis_set=dm.select(datasets=datasets or DATASETS, variables=variables),
                        filters=filters, handle_missing=handler)


def labels_of(variable):
    """{code: label} of a categorical variable, whatever format the catalogue uses."""
    raw = variable._data.get("enumerations") or {}
    if isinstance(raw, dict):
        return {str(k): str(v) for k, v in raw.items()}
    return {str(item.get("code") if isinstance(item, dict) else item):
            str(item.get("label") if isinstance(item, dict) else item) for item in raw}


def rows_for(result, variable, dataset=POOLED):
    """The describe row of one variable: the pooled row by default, or the row of one Dataset."""
    keys = (POOLED,) if dataset == POOLED else (dataset._code, dataset.label)
    return [r for r in result.raw.get("featurewise", [])
            if r.get("variable") == variable._code and r.get("dataset") in keys]


def numeric_summary(result, variables, dataset=POOLED):
    """n, missing, mean, SD, quartiles and range of numerical variables from a describe result."""
    rows = []
    for v in variables:
        for r in rows_for(result, v, dataset):
            d = r.get("data") or {}
            rows.append({"variable": v.label, "n": d.get("num_dtps"), "missing": d.get("num_na"),
                         "mean": d.get("mean"), "sd": d.get("std"), "q1": d.get("q1"), "median": d.get("q2"),
                         "q3": d.get("q3"), "min": d.get("min"), "max": d.get("max")})
    return pd.DataFrame(rows)


def category_counts(result, variable, dataset=POOLED):
    """Counts and percentages of the levels of a categorical variable from a describe result."""
    lab = labels_of(variable)
    rows = []
    for r in rows_for(result, variable, dataset):
        counts = (r.get("data") or {}).get("counts") or {}
        total = sum(counts.values())
        for c, n in counts.items():
            rows.append({"variable": variable.label, "level": lab.get(str(c), str(c)), "count": n,
                         "percent": 100 * n / total if total else np.nan, "missing": (r.get("data") or {}).get("num_na")})
    return pd.DataFrame(rows)


def level_percent(result, variable, levels=("Yes",), dataset=POOLED):
    """Percentage of the given levels (labels) of a categorical variable in a describe result."""
    k = category_counts(result, variable, dataset)
    return float(k.loc[k["level"].isin(levels), "percent"].sum()) if len(k) else np.nan


def by_level(numeric, group, levels=None, base_filter=None, datasets=None):
    """n, mean, median and quartiles of a numerical variable in each level of a categorical one
    (one describe per level). levels: the codes to use, by default every code of the variable."""
    lab = labels_of(group)
    rows = []
    for c in levels or lab:
        f = F(group) == c
        if base_filter is not None:
            f = f & base_filter
        res = analysis([numeric], f, datasets=datasets).describe()
        for r in rows_for(res, numeric):
            d = r.get("data") or {}
            rows.append({group.label: lab.get(c, c), "n": d.get("num_dtps"), "mean": d.get("mean"),
                         "q1": d.get("q1"), "median": d.get("q2"), "q3": d.get("q3")})
    return pd.DataFrame(rows)


def band_means(numeric, bands, base_filter=None, datasets=None):
    """n, mean and median of a numerical variable in bands given as {label: filter}."""
    rows = []
    for label, f in bands.items():
        if base_filter is not None:
            f = f & base_filter
        res = analysis([numeric], f, datasets=datasets).describe()
        for r in rows_for(res, numeric):
            d = r.get("data") or {}
            rows.append({"band": label, "n": d.get("num_dtps"), "mean": d.get("mean"), "median": d.get("q2")})
    return pd.DataFrame(rows)


def crosstab(exposure, outcome, base_filter=None, event="1", levels=None, event_label=None, datasets=None):
    """Cross-tabulation of outcome by exposure, as the MIP does it: counts come from one
    describe per exposure level, the p-value from the Chi-squared Test on the whole cohort.
    `event` is the outcome code counted in the percentage column, or a list of codes."""
    lab_x, lab_y = labels_of(exposure), labels_of(outcome)
    events = [event] if isinstance(event, str) else list(event)
    pct_col = "% " + (event_label or lab_y.get(events[0], events[0]))
    rows = []
    for c in levels or lab_x:
        f = (F(exposure) == c) & F(outcome).is_not_null()
        if base_filter is not None:
            f = f & base_filter
        res = analysis([exposure, outcome], f, datasets=datasets).describe()
        counts = {}
        for r in rows_for(res, outcome):
            counts = (r.get("data") or {}).get("counts") or {}
        n = sum(counts.values())
        if n == 0:
            continue
        row = {exposure.label: lab_x.get(c, c), "n": n}
        for yc, ylabel in lab_y.items():
            row[ylabel] = counts.get(yc, 0)
        row[pct_col] = 100 * sum(counts.get(e, 0) for e in events) / n
        rows.append(row)
    table = pd.DataFrame(rows)
    f = F(exposure).is_not_null() & F(outcome).is_not_null()
    if levels:
        f = f & F(exposure).isin(list(levels))
    if base_filter is not None:
        f = f & base_filter
    chi = analysis([exposure, outcome], f, drop_missing=True, datasets=datasets).chi_square_test(x=exposure, y=outcome)
    stats = chi.raw if isinstance(chi.raw, dict) else {}
    table.attrs["chi2"] = stats.get("chi2")
    table.attrs["p"] = stats.get("p_value", stats.get("p"))
    table.attrs["outcome"] = outcome.label
    return table


def rates_by(group, outcomes, levels=None, base_filter=None, event="1", datasets=None):
    """% of records with each outcome (code `event`) in every level of `group`: one describe per
    level for the counts of all outcomes, one Chi-squared Test per outcome (in .attrs)."""
    lab = labels_of(group)
    rows = []
    for c in levels or lab:
        f = F(group) == c
        if base_filter is not None:
            f = f & base_filter
        res = analysis(outcomes, f, datasets=datasets).describe()
        row = {group.label: lab.get(c, c)}
        for y in outcomes:
            counts = {}
            for r in rows_for(res, y):
                counts = (r.get("data") or {}).get("counts") or {}
            n = sum(counts.values())
            row.setdefault("n", n)
            row[y.label] = 100 * counts.get(event, 0) / n if n else np.nan
        rows.append(row)
    table = pd.DataFrame(rows)
    table.attrs["chi2"], table.attrs["p"] = {}, {}
    for y in outcomes:
        f = F(group).is_not_null() & F(y).is_not_null()
        if levels:
            f = f & F(group).isin(list(levels))
        if base_filter is not None:
            f = f & base_filter
        chi = analysis([group, y], f, drop_missing=True, datasets=datasets).chi_square_test(x=group, y=y)
        table.attrs["chi2"][y.label] = chi.raw.get("chi2")
        table.attrs["p"][y.label] = chi.raw.get("p_value")
    return table


def month_label(i):
    """Month index 1..36 -> '2020-01' .. '2022-12'."""
    return f"{2020 + (i - 1) // 12}-{(i - 1) % 12 + 1:02d}"


def by_month(outcomes, variable, months=range(1, 37), datasets=None):
    """% of records with each outcome (code 1) per month of the time axis: one describe per month."""
    rows = []
    for m in months:
        res = analysis(outcomes, F(variable) == m, datasets=datasets).describe()
        row = {"month": m, "label": month_label(m)}
        for y in outcomes:
            counts = {}
            for r in rows_for(res, y):
                counts = (r.get("data") or {}).get("counts") or {}
            n = sum(counts.values())
            row.setdefault("n", n)
            row[y.label] = 100 * counts.get("1", 0) / n if n else np.nan
        rows.append(row)
    return pd.DataFrame(rows)


def fmt_p(p):
    if p is None or (isinstance(p, float) and np.isnan(p)):
        return "n/a"
    return "< 0.001" if p < 0.001 else f"{p:.3f}"


def show_crosstab(table, title=None, rotate=0):
    """Display a crosstab table, its chi-square test and a bar chart of the percentages."""
    pct_col = [c for c in table.columns if c.startswith("% ")][0]
    display(table.style.format({pct_col: PCT}).hide(axis="index"))
    print(f"Chi-squared Test: chi2 = {table.attrs['chi2']:,.0f}, p = {fmt_p(table.attrs['p'])}")
    ax = table.set_index(table.columns[0])[pct_col].plot(kind="bar", color="#4C72B0", figsize=(7, 3))
    ax.set_ylabel(pct_col)
    ax.set_title(title or f"{table.attrs['outcome']} by {table.columns[0]}")
    ax.set_xlabel("")
    plt.xticks(rotation=rotate)
    plt.tight_layout()
    plt.show()


def anova_summary(result):
    """One-way ANOVA result -> F, p, n, a table of the group means and SDs, and the Tukey comparisons."""
    raw = result.raw if isinstance(result.raw, dict) else {}
    table, ci = raw.get("anova_table", {}), raw.get("ci_info", {})
    groups = pd.DataFrame({"mean": pd.Series(ci.get("means", {})), "sd": pd.Series(ci.get("sample_stds", {}))})
    return table.get("f_stat"), table.get("p_value"), table.get("n_obs"), groups, pd.DataFrame(raw.get("tuckey_test", []))


def ttest_summary(result):
    """Independent t-test result -> t, p, df, mean difference with its 95% CI, Cohen's d, n."""
    raw = result.raw if isinstance(result.raw, dict) else {}
    return {k: raw.get(k) for k in ("t_stat", "p", "df", "mean_diff", "ci_lower", "ci_upper", "cohens_d", "n_obs")}


def n_fmt(value):
    # a count with thousands separators, or n/a when the platform did not return it
    return f"{value:,}" if isinstance(value, (int, float)) else "n/a"


def ttest_from_describe(result_a, result_b, variable):
    """The pooled-variance t statistic of two groups from their describe results (n, mean, SD):
    all a t-test needs, for groups the platform's t-test cannot form (several codes pooled)."""
    a = numeric_summary(result_a, [variable]).iloc[0]
    b = numeric_summary(result_b, [variable]).iloc[0]
    n1, n2 = a["n"], b["n"]
    sp2 = ((n1 - 1) * a["sd"] ** 2 + (n2 - 1) * b["sd"] ** 2) / (n1 + n2 - 2)
    t = (a["mean"] - b["mean"]) / math.sqrt(sp2 * (1 / n1 + 1 / n2))
    return {"mean_a": a["mean"], "mean_b": b["mean"], "diff": a["mean"] - b["mean"], "t": t, "n": n1 + n2}


def term_label(term, variables):
    """Model term -> readable label: 'SEX[2]' -> 'Sex = Female', 'AGE' -> 'Age (per unit)'."""
    by_code = {v._code: v for v in variables}
    term = str(term)
    if "[" in term and term.endswith("]"):
        base, level = term[:-1].split("[", 1)
        v = by_code.get(base)
        return f"{v.label} = {labels_of(v).get(level, level)}" if v else term
    v = by_code.get(term)
    return f"{v.label} (per unit)" if v else term


def odds_ratio_table(result, variables):
    """Logistic regression result -> one row per term with odds ratio, 95% CI and p.
    Categorical covariates appear as 'variable = level' relative to the reference level
    (the first code of the variable) that the platform leaves out."""
    frame = result.to_frame().copy()
    frame = frame[~frame["feature"].astype(str).str.lower().isin(["intercept", "const"])]
    frame["term"] = frame["feature"].map(lambda t: term_label(t, variables))
    for col in ("odds_ratio", "or_lower_ci", "or_upper_ci", "p"):
        if col not in frame.columns:
            frame[col] = np.nan
    out = frame[["term", "odds_ratio", "or_lower_ci", "or_upper_ci", "p"]]
    return out.rename(columns={"or_lower_ci": "ci_low", "or_upper_ci": "ci_high"}).reset_index(drop=True)


def show_odds_ratios(table, title):
    """Table and forest plot (log scale) of odds ratios."""
    display(table.style.format({"odds_ratio": "{:.3f}", "ci_low": "{:.3f}", "ci_high": "{:.3f}", "p": "{:.4f}"}).hide(axis="index"))
    t = table.iloc[::-1]
    fig, ax = plt.subplots(figsize=(8, 0.5 * len(t) + 1.5))
    ax.errorbar(t["odds_ratio"], range(len(t)),
                xerr=[t["odds_ratio"] - t["ci_low"], t["ci_high"] - t["odds_ratio"]], fmt="o", capsize=4)
    ax.axvline(1, linestyle="--", linewidth=1)
    ax.set_xscale("log")
    ax.set_yticks(range(len(t)))
    ax.set_yticklabels(t["term"])
    ax.set_xlabel("odds ratio (log scale)")
    ax.set_title(title)
    plt.tight_layout()
    plt.show()


def linear_table(result, variables):
    """Linear regression result -> one row per term with coefficient (units of the outcome), 95% CI and p."""
    raw = result.raw if isinstance(result.raw, dict) else {}
    return pd.DataFrame({"term": [term_label(t, variables) for t in raw.get("indep_vars", [])],
                         "coefficient": raw.get("coefficients"), "ci_low": raw.get("lower_ci"),
                         "ci_high": raw.get("upper_ci"), "p": raw.get("pvalues")})


def show_coefficients(table, title, unit="days"):
    """Table and dot chart of linear-regression coefficients (the intercept left out of the chart)."""
    display(table.style.format({"coefficient": "{:.3f}", "ci_low": "{:.3f}", "ci_high": "{:.3f}", "p": "{:.4f}"}).hide(axis="index"))
    t = table[table["term"] != "Intercept"].iloc[::-1]
    fig, ax = plt.subplots(figsize=(8, 0.5 * len(t) + 1.5))
    ax.errorbar(t["coefficient"], range(len(t)),
                xerr=[t["coefficient"] - t["ci_low"], t["ci_high"] - t["coefficient"]], fmt="o", capsize=4)
    ax.axvline(0, linestyle="--", linewidth=1)
    ax.set_yticks(range(len(t)))
    ax.set_yticklabels(t["term"])
    ax.set_xlabel(f"{unit} added, for the same values of the other predictors")
    ax.set_title(title)
    plt.tight_layout()
    plt.show()


print("Toolbox ready.")

## 3. The dataset and the variables of this scenario

| Code | Label in the MIP | Tree location (under NHCS) | Role | Values |
|---|---|---|---|---|
| `NEWBORN` | Newborn | Patient | cohort filter: no newborn stays | 1 Newborn; 2 Not a newborn |
| `died` | Died during encounter | Discharge | outcome; curation-derived | 0 No; 1 Yes |
| `YEAR` | Survey year | Encounter | mortality by year | 2020 2020; 2021 2021; 2022 2022 |
| `SEX` | Sex | Patient | predictor | 1 Male; 2 Female |
| `AGE` | Age | Patient | predictor, continuous; t-test | integer, 0-86 years |
| `age_group` | Age group | Patient | mortality by age; curation-derived | 1 Under 1 year; 2 1-17 years; 3 18-44 years; 4 45-64 years; 5 65-74 years; 6 75-84 years; 7 85 years and over |
| `charlson_group` | Charlson index group | Diagnoses / Condition flags (curated) | predictor: comorbidity; curation-derived | 0 0 (no Charlson condition); 1 1-2; 2 3-4; 3 5 or more |
| `covid_dx` | COVID-19, any position | Diagnoses / Condition flags (curated) | predictor; curation-derived | 0 No; 1 Yes |
| `covid_dx1` | COVID-19 as first-listed diagnosis | Diagnoses / Condition flags (curated) | extension; curation-derived | 0 No; 1 Yes |
| `sepsis_dx` | Sepsis, any position | Diagnoses / Condition flags (curated) | predictor; curation-derived | 0 No; 1 Yes |
| `dx1_group` | First-listed diagnosis, group | Diagnoses | mortality by reason for admission; curation-derived | 40 codes, listed in the tree |

Variables are picked by their **label**, as the tree shows it; the tree location tells you where
to find them in `dm.tree()` and in the MIP interface. The code (for example `AGE`) is what the csv
column is called; you will see it in some result payloads. The first line below chooses the
dataset(s) every call of this notebook uses.

In [ ]:
DATASETS = [ip]                          # every call of this notebook runs on the inpatient discharges

newborn    = dm.variables["Newborn"]
died       = dm.variables["Died during encounter"]
year       = dm.variables["Survey year"]
sex        = dm.variables["Sex"]
age        = dm.variables["Age"]
age_group  = dm.variables["Age group"]
charlson   = dm.variables["Charlson index group"]
covid      = dm.variables["COVID-19, any position"]
covid1     = dm.variables["COVID-19 as first-listed diagnosis"]
sepsis     = dm.variables["Sepsis, any position"]
group      = dm.variables["First-listed diagnosis, group"]
cohort = F(newborn) == "2"               # no newborn stays, for every step
display(mip.to_frame([newborn, died, year, sex, age, age_group, charlson, covid, covid1, sepsis, group]))

## Step 1. The outcome

Deaths among all discharges, among newborn stays, and in the cohort without newborns; then by
year.

*Expected:* all discharges: 12,560 deaths among the 478,347 with a known discharge status
(2.6%; the status is missing for 10,289); newborn stays 0.5%; without newborns 12,316 deaths
among 427,434 (2.9%). By year 3.1% in 2020, 3.0% in 2021, 2.7% in 2022 (chi-square 49,
p < 0.001).

In [ ]:
for label, f in (("all discharges", None), ("newborn stays", F(newborn) == "1"), ("without newborns", cohort)):
    res = analysis([died], f).describe()
    k = category_counts(res, died)
    print(f"{label}: {int(k.loc[k['level'] == 'Yes', 'count'].sum()):,} deaths among {int(k['count'].sum()):,} "
          f"with a known discharge status ({level_percent(res, died):.2f}%; status missing for {int(k['missing'].iloc[0]):,})")
show_crosstab(crosstab(year, died, base_filter=cohort, event_label="died"), "In-hospital mortality by year, newborns excluded")

## Step 2. Who dies: age, sex, comorbidity

*Expected:* men 3.6%, women 2.3%; under 1 year 0.8%, 1 to 17 years 0.4%, 18 to 44 years 0.8%,
45 to 64 years 2.8%, 65 to 74 years 4.0%, 75 to 84 years 4.9%, 85 and over 6.4%; Charlson index
0: 1.3%, 1-2: 2.8%, 3-4: 4.2%, 5 or more: 5.7%. All p < 0.001.

In [ ]:
factors = []
for exposure in (sex, age_group, charlson):
    t = crosstab(exposure, died, base_filter=cohort, event_label="died")
    print(f"{exposure.label}: chi2 = {t.attrs['chi2']:,.0f}, p = {fmt_p(t.attrs['p'])}")
    for _, r in t.iterrows():
        factors.append({"factor": exposure.label, "level": r[exposure.label], "n": r["n"], "% died": r["% died"]})
factors = pd.DataFrame(factors)
display(factors.style.format({"% died": PCT}).hide(axis="index"))
fig, ax = plt.subplots(figsize=(8, 4))
labels = factors["factor"] + ": " + factors["level"].astype(str)
ax.barh(labels[::-1], factors["% died"][::-1], color="#C44E52")
ax.set_xlabel("% died during the stay"); ax.set_title("In-hospital mortality by sex, age group and Charlson index")
plt.tight_layout()
plt.show()

## Step 3. Why they were admitted

Mortality by COVID-19 and sepsis among the diagnoses, and by the twelve most frequent first-listed
diagnosis groups (the `levels` argument limits the cross-tabulation to them).

*Expected:* COVID-19 among the diagnoses 9.9% against 2.4%; sepsis 14.9% against 1.7%. By
first-listed diagnosis: sepsis 12.4%, COVID-19 9.6%, cerebrovascular disease 5.6%, neoplasms
3.4%, other circulatory 2.8%, hypertensive diseases 2.7%, poisoning and complications of care
2.4%, digestive 1.9%, fracture 1.2%, musculoskeletal 0.5%, other mental disorders 0.1%,
pregnancy 0.0%. The reason for admission spreads mortality from near zero to one in eight.

In [ ]:
for exposure in (covid, sepsis):
    t = crosstab(exposure, died, base_filter=cohort, event_label="died")
    print(f"{exposure.label}: " + ", ".join(f"{r[exposure.label]} {r['% died']:.1f}%" for _, r in t.iterrows()) + f"; chi2 = {t.attrs['chi2']:,.0f}, p = {fmt_p(t.attrs['p'])}")

top12 = ["2", "26", "3", "6", "38", "28", "15", "20", "12", "19", "36", "4"]    # the most frequent groups, newborns excluded
by_reason = crosstab(group, died, base_filter=cohort, levels=top12, event_label="died").sort_values("% died", ascending=False)
display(by_reason[[group.label, "n", "% died"]].style.format({"% died": PCT}).hide(axis="index"))
print(f"Chi-squared Test over the twelve groups: chi2 = {by_reason.attrs['chi2']:,.0f}, p = {fmt_p(by_reason.attrs['p'])}")
short = by_reason[group.label].str.replace(r" \(.*\)", "", regex=True)
ax = pd.Series(by_reason["% died"].values, index=short).iloc[::-1].plot(kind="barh", color="#C44E52", figsize=(8, 4.5))
ax.set_xlabel("% died during the stay"); ax.set_title("In-hospital mortality by first-listed diagnosis group")
plt.tight_layout()
plt.show()

## Step 4. The case fatality of COVID-19, year by year

Mortality by year among stays with and without a COVID-19 diagnosis.

*Expected:* with COVID-19, 13.7% died in 2020 (6,367 stays with a known status), 11.4% in 2021
(9,590), 6.1% in 2022 (10,011); without, 2.5%, 2.4% and 2.4%. In two years the case fatality of a COVID-19 stay
fell by more than half while the number of such stays rose. Vaccination, better treatment, a
milder variant and a change in who was admitted (and in who was coded) all contribute; the data
cannot separate them.

In [ ]:
with_covid = crosstab(year, died, base_filter=cohort & (F(covid) == "1"), event_label="died").set_index("Survey year")
without = crosstab(year, died, base_filter=cohort & (F(covid) == "0"), event_label="died").set_index("Survey year")
fatality = pd.DataFrame({"stays with COVID-19": with_covid["n"], "% died, with COVID-19": with_covid["% died"], "% died, without": without["% died"]})
display(fatality.style.format({"% died, with COVID-19": PCT, "% died, without": PCT}))
ax = fatality[["% died, with COVID-19", "% died, without"]].plot(kind="bar", figsize=(6, 3.2), rot=0, color=["#C44E52", "#4C72B0"])
ax.set_ylabel("% died during the stay"); ax.set_xlabel(""); ax.set_title("Case fatality of stays with and without COVID-19")
ax.legend(frameon=False)
plt.tight_layout()
plt.show()

## Step 5. Age, as a mean

*Expected:* 68.1 years in patients who died against 53.8 in those discharged alive, a difference
of 14.4 years; t about 70, p < 0.001.

In [ ]:
t = analysis([age, died], cohort, drop_missing=True).t_test(variable=age, group_by=died, group_a="1", group_b="0")
s = ttest_summary(t)
print(f"Age, died minus discharged alive: {s['mean_diff']:.1f} years (95% CI {s['ci_lower']:.1f} to {s['ci_upper']:.1f}), "
      f"t = {s['t_stat']:.1f}, p = {fmt_p(s['p'])}, n = {n_fmt(s['n_obs'])}")

## Step 6. The model

A *Logistic Regression* of death on age, sex, Charlson group, COVID-19 and sepsis, newborns
excluded. The platform compares each category with its first code: men, Charlson 0, no COVID-19,
no sepsis.

*Expected (427,100 discharges with a known status, age, sex and condition flags), odds of dying:*
age 1.028 per year (1.32 per 10 years); women
0.80 (0.77 to 0.83); Charlson 1-2 1.23, 3-4 1.65, 5 or more 2.26 (versus 0); COVID-19 3.19 (3.04
to 3.35); sepsis 7.43 (7.15 to 7.72). Sepsis multiplies the odds of death by seven and COVID-19
by three, for the same age, sex and comorbidity; each decade adds a third; women fare better than
men at the same age and comorbidity, as in most mortality data.

In [ ]:
covariates = [age, sex, charlson, covid, sepsis]
t0 = time.perf_counter()
model = analysis([died] + covariates, cohort, drop_missing=True).logistic_regression(x=covariates, y=died, positive_class="1")
print(f"{model.raw['summary']['n_obs']:,} discharges in the model; the platform answered in {time.perf_counter() - t0:.0f} s")
ors = odds_ratio_table(model, covariates)
show_odds_ratios(ors, "Odds of dying during the stay, sampled discharges 2020 to 2022, newborns excluded")
per_year = ors.set_index("term").loc["Age (per unit)"]
print(f"Age, per 10 years: {per_year['odds_ratio'] ** 10:.2f}")

## Step 7. State the result

A model answer:

> Among 437,484 sampled US hospital discharges of 2020 to 2022 (newborns excluded), 2.8% ended in
> death, falling from 3.1% in 2020 to 2.7% in 2022. Mortality rose with age (1.32 per 10 years),
> comorbidity (2.3 for a Charlson index of 5 or more) and above all with sepsis (odds ratio 7.4)
> and COVID-19 (3.2). The case fatality of stays with COVID-19 fell from 13.7% in 2020 to 6.1% in
> 2022. These are unweighted sample statistics, and death after discharge is not recorded.

**Questions for discussion.** The yearly mortality fell from 3.1% to 2.7%: how much of that is
COVID-19? Women have a lower odds of dying at the same age and Charlson index: biology, coding, or
who gets admitted? What happens to a comorbidity index when a hospital codes few diagnoses? Would
you use this model to compare the mortality of two hospitals?

**Optional extension.** Add the survey year to the model: with age, sex, comorbidity, sepsis and
COVID-19 held constant, is 2022 still safer than 2020? Then replace COVID-19 in any position by
COVID-19 as the first-listed diagnosis.

In [ ]:
for covariates in ([age, sex, charlson, covid, sepsis, year], [age, sex, charlson, covid1, sepsis]):
    model = analysis([died] + covariates, cohort, drop_missing=True).logistic_regression(x=covariates, y=died, positive_class="1")
    display(odds_ratio_table(model, covariates).style.format({"odds_ratio": "{:.3f}", "ci_low": "{:.3f}", "ci_high": "{:.3f}", "p": fmt_p}).hide(axis="index"))